In [8]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,text,label
0,139768,Take a Presence Power Break (The New Coffee Br...,WELLNESS
1,297,Yolanda Hadid Returns To Social Media After 9-...,ENTERTAINMENT
2,2274,"Democrats Want Paid Sick Days, Breaks For Dome...",POLITICS
3,106057,Taylor Swift Calls Out Sexist Critics Again,ENTERTAINMENT
4,56920,Trump Surrogate Rudy Giuliani: 'Anything's Leg...,POLITICS


In [9]:
df_train.text[6]

"Transform While There's Still Time The mission is to prepare people before their world falls apart, so they won't be surprised and fall into panic and traumatic depression. Carolyn Baker's dream is that a better society will arise as we wisely prepare for and resiliently move through a period of suffering."

In [10]:
#preprocessinguhh
import spacy
import re

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', "", text)
    text = re.sub(r'[\d+]', "", text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(tokens)

preprocess(df_train.text[6])

'transform s time mission prepare people world fall apart will not surprised fall panic traumatic depression carolyn baker dream well society arise wisely prepare resiliently period suffering'

In [11]:
df_train.text = df_train.text.apply(preprocess)
df_test.text = df_test.text.apply(preprocess)

df_train.head()

,SampleID,text,label
0,139768,presence power break new coffee break soon hon...,WELLNESS
1,297,yolanda hadid return social medium month break...,ENTERTAINMENT
2,2274,democrats want pay sick day break domestic wor...,POLITICS
3,106057,taylor swift call sexist critic,ENTERTAINMENT
4,56920,trump surrogate rudy giuliani anything legal w...,POLITICS


In [16]:
X = df_train.text
y = df_train.label.map({"WELLNESS": 1, "ENTERTAINMENT": 2, "POLITICS": 3})

X_final = df_test.text

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer()

X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X

<56727x41302 sparse matrix of type '<class 'numpy.float64'>'
	with 825311 stored elements in Compressed Sparse Row format>

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape

(45381, 41302)

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

model = LogisticRegression().fit(X_train, y_train)
y_preds = model.predict(X_valid)
f1_score(y_valid, y_preds, average="macro")

0.9275210343857707

In [34]:
y_final = model.predict(X_final)

In [35]:
submission = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "label": y_final
})

submission.label = submission.label.map({1: "WELLNESS", 2: "ENTERTAINMENT", 3: "POLITICS"})

submission

,SampleID,label
0,103811,POLITICS
1,207637,WELLNESS
2,151717,WELLNESS
3,27926,POLITICS
4,115640,POLITICS
...,...,...
14177,47587,ENTERTAINMENT
14178,38503,POLITICS
14179,195242,WELLNESS
14180,207267,WELLNESS


In [36]:
submission.to_csv("submission.csv", index=False)